# Demo 3 — Strands Agents: The Framework Does the Loop

Demo 2's ReAct loop was ~40 lines of hand-written orchestration.
**Strands Agents** (AWS open source) reduces it to: *model + tools + prompt*.

- `@tool` decorator turns any Python function into a tool
  (schema auto-generated from the signature + docstring)
- The agent loop, retries, streaming, and tracing come for free

In [1]:
from strands import Agent, tool
from strands.models import BedrockModel

## Same travel toolbox as Demo 2 — but now just decorated functions

In [2]:
@tool
def list_cities_on_route(route: str) -> dict:
    """List the major cities along a cycling route, in order.

    Args:
        route: Route name, e.g. 'berlin-munich'
    """
    routes = {"berlin-munich": ["Berlin", "Leipzig", "Nuremberg", "Munich"]}
    return {"cities": routes.get(route.lower(), [])}


@tool
def get_weather(city: str) -> dict:
    """Get current weather for a city.

    Args:
        city: City name
    """
    fake_db = {
        "berlin": {"temp_c": 22, "condition": "sunny"},
        "leipzig": {"temp_c": 21, "condition": "cloudy"},
        "nuremberg": {"temp_c": 17, "condition": "rain"},
        "munich": {"temp_c": 18, "condition": "rain"},
    }
    return fake_db.get(city.lower(), {"temp_c": 20, "condition": "unknown"})

## Assemble the agent — this is the whole thing

In [3]:
model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name="us-west-2",
)

agent = Agent(
    model=model,
    system_prompt="You are a cycling trip assistant. Be concise.",
    tools=[list_cities_on_route, get_weather],
)

## Run it — same question as Demo 2

In [4]:
result = agent(
    "I'm cycling the berlin-munich route. Which cities on the route "
    "will I need rain gear in, based on current weather?"
)

I'll help you check the weather along the berlin-munich route. Let me first get the cities on the route, then

 check the weather for each.
Tool #1: list_cities_on_route


Now let me check the current weather for each of these cities:
Tool #2: get_weather



Tool #3: get_weather

Tool #4: get_weather



Tool #5: get_weather


Based on current weather along the berlin-munich route, you'll need rain

 gear in:

- **Nuremberg** (rain, 17°C)
- **

Munich** (rain, 18°C)

Berlin is sunny (22°C) and Leipzig is cloudy (21°C), so you won

't need rain gear at the start of your trip, but bring it for the southern part of your route

!

## Inspect what happened: the tool-call trace

In [5]:
for msg in agent.messages:
    for block in msg["content"]:
        if "toolUse" in block:
            tu = block["toolUse"]
            print(f"→ {tu['name']}({tu['input']})")

→ list_cities_on_route({'route': 'berlin-munich'})
→ get_weather({'city': 'Berlin'})
→ get_weather({'city': 'Leipzig'})
→ get_weather({'city': 'Nuremberg'})
→ get_weather({'city': 'Munich'})


## Takeaways

- The hand-written loop from Demo 2 became **zero lines** — Strands owns it
- Tools are plain Python functions; the docstring *is* the API contract
- Full message history stays inspectable (`agent.messages`) — no black box
- Next: a different philosophy — **CrewAI**, where multiple specialized
  agents collaborate on a task